# Making Roman image cutouts with Gladius

This notebook opens Gladius in your browser. You can choose images on your
computer or find images in the MAST archive, add a list of targets, and make
cutouts around them. The form explains the settings and shows any errors.

The interface needs no MAST token when you use local images. Archive access
uses a token for your MAST account. A separate example later in this notebook
shows how to query the archive and review cutouts from Python.


In [ ]:
from pathlib import Path

# Keep configurations, job records, and example outputs in this folder.
WORKSPACE = Path.cwd() / "gladius-archive-review"

# Open the browser interface when its launch cell runs.
RUN_GUI = True

# Match the scheme, hostname, and port in your JupyterLab address.
NOTEBOOK_ORIGIN = "http://localhost:8888"

# Optional for MAST; local images need no credential file.
ENV_FILE = None

# ENV_FILE = Path("/path/to/.env")


## Opening Gladius

The next cell opens the interface inside this notebook and prints a link for
opening it in a separate tab. Gladius reads and writes files on the computer
running the notebook. `WORKSPACE` above chooses its working folder.

For the embedded interface, `NOTEBOOK_ORIGIN` above needs the beginning of your
JupyterLab address, such as `http://localhost:8888`. It includes the scheme,
hostname, and port, but no path or login token.

For MAST access, `ENV_FILE` above can point to your credential file. Gladius also
reads `.env` in `WORKSPACE` and uses any token already in the kernel’s environment.
Missing credentials do not prevent the interface from opening. If a query needs
authorization, Gladius reports the archive error in the interface when you try it.


In [ ]:
if RUN_GUI:
    import threading
    from IPython.display import IFrame, display
    from gladius.web import GuiServer

    from gladius import load_environment

    # Create the working folder before starting the server.
    WORKSPACE.mkdir(parents=True, exist_ok=True)

    # Load optional credentials without preventing local-file use if they are unavailable.
    if ENV_FILE is not None:
        try:
            load_environment(env_file=ENV_FILE)
        except (OSError, ValueError):
            print("The credential file could not be loaded. Local files still work; MAST access may fail.")

    # Reuse a running server when this cell runs again.
    # A background thread keeps the interface responsive while other cells run.
    if "gui_thread" not in globals() or not gui_thread.is_alive():
        gui_server = GuiServer(WORKSPACE, port=0, iframe_origin=NOTEBOOK_ORIGIN, initial_config=globals().get("config"))
        gui_thread = threading.Thread(target=gui_server.serve_forever, daemon=True)
        gui_thread.start()

    # Show the interface inside the notebook and provide a separate-tab link.
    gladius_ui_url = gui_server.url + "/"
    print("Open the GUI in a separate tab:", gladius_ui_url)
    display(IFrame(src=gladius_ui_url, width="100%", height=650))
else:
    print("GUI skipped. Set RUN_GUI=True in the settings cell to explore it.")


## Stopping Gladius

Gladius keeps running when you close its frame or browser tab. Once your job
finishes, you can stop it by changing `STOP_GUI` to `True` and running the next
cell. Leaving it `False` keeps the interface open when you run the whole notebook.


In [ ]:
STOP_GUI = False  # True stops the interface after any active job finishes.

if STOP_GUI:
    if "gui_thread" in globals() and gui_thread.is_alive():

        # Stop accepting requests, release the port, and wait for the server thread.
        gui_server.shutdown()
        gui_server.server_close()
        gui_thread.join(timeout=5)
        print("Embedded GUI stopped.")


## Archive example

This section shows how to make cutouts from MAST using Python. It is separate
from the interface above, so you can skip it when you only want to use the form.

The example reads a target CSV and a configuration, then runs Gladius. Archive
access uses the credentials loaded above. Missing files, invalid settings, or
archive access problems produce an error when the relevant cell runs.


### A target list

The next cell creates `targets.csv` in the workspace from the rows shown in
`TARGETS_CSV`. You can edit those rows in the notebook. Each row gives a target
name, right ascension, and declination in ICRS degrees.

The example uses the approximate ICRS position of Sagittarius A*. The target
ID `sgr-a-star` gives its output files a filesystem-safe name. Running the cell again replaces this example CSV with your edited rows.


In [ ]:
# Sagittarius A*: approximate ICRS position in degrees.
TARGETS_CSV = """id,ra,dec
sgr-a-star,266.416837,-29.007810
"""

# Save the rows above as a file that Gladius can read. Rerunning replaces this example file.
TARGET_CSV = WORKSPACE / "targets.csv"
WORKSPACE.mkdir(parents=True, exist_ok=True)
TARGET_CSV.write_text(TARGETS_CSV)


### Choosing the cutouts

The configuration below asks MAST for calibrated images near the targets and
makes 150 × 150 pixel cutouts. A six-arcminute search radius finds candidate
images; Gladius skips a target if it falls outside a detector.

The example saves ASDF and FITS cutouts and keeps the original images for the
previews below. Keeping full images can use substantial disk space. Date,
program, detector, and filter selections can narrow a larger search; the
[configuration guide](https://roman-gladius.readthedocs.io/en/latest/configuration.html)
explains those options.

The quality screen and display products start disabled. The later sections show
how to inspect the cutouts before choosing those settings.


In [ ]:
import json
from gladius import RunConfig, read_targets_csv, run

# Describe the run first; creating this object does not query or download anything.
config = RunConfig(
    run={"name": "archive-field-review"},

    # Read target names and coordinates from the CSV we just created.
    targets=read_targets_csv(TARGET_CSV),

    # Search MAST near each target; keep at most one observation for this example.
    inputs={"mode": "mast", "mast_radius_arcmin": 6, "query_limit": 1},

    # Keep exposures starting on or after October 1 and before November 1, UTC.
    selection={"start": "2026-10-01T00:00:00", "end": "2026-11-01T00:00:00"},

    # Copy a 150-pixel square, padding any part beyond the detector edge.
    extraction={"size": 150, "edge": "pad"},

    # Check that the coordinate model maps pixels to the sky and back consistently.
    astrometry={"mode": "validate", "roundtrip_tolerance_pixels": 0.01},

    # Leave the optional elongation screen off until we inspect the results.
    quality={"autocorrelation_limit": None, "tile_size": 64, "radius": 15},

    # Start with individual cutouts; later cells add display products.
    stacks={"enabled": False, "gif_enabled": False, "rgb_enabled": False},

    # Save both cutout formats and retain full images for context previews.
    storage={"output_directory": "output", "retain_full_images": True,
             "formats": ["asdf", "fits"]},

# Resolve relative file paths from the workspace and fill in default settings.
).resolved(WORKSPACE)


### Reviewing the search and settings

The next cell prints the queries Gladius will send and saves the configuration
as YAML. It does not contact MAST. A target list can select many full detector
images, even when each requested cutout is small. The query limit selects a limited sample, not a complete inventory. It does
not impose a download-size limit.


In [ ]:
import yaml
from gladius.inputs import mast_queries

# Build the archive searches from our targets and settings.
# This only describes the searches; it does not contact MAST.
queries = mast_queries(config)
for query in queries:
    print(query)

# Convert the configuration object to a dictionary of ordinary values.
# JSON mode turns objects such as file paths into text that YAML can store.
settings = config.model_dump(mode="json")

# Turn that dictionary into readable YAML, keeping the settings in their original order.
configuration_yaml = yaml.safe_dump(settings, sort_keys=False)
print(configuration_yaml)

# Create the workspace if needed, then save the same YAML shown above.
# This file lets us repeat the run from the command line or load it into the UI.
WORKSPACE.mkdir(parents=True, exist_ok=True)
configuration_path = WORKSPACE / "review-config.yaml"
configuration_path.write_text(configuration_yaml)
print("Saved configuration:", configuration_path)


### Making cutouts

The next cell queries MAST, downloads the selected images, and makes cutouts.
Gladius opens each image once for all targets. This configuration keeps those
images for context previews. Matching cached work can be reused; the progress
messages and manifest record what Gladius did.

If MAST denies access, this cell offers a hidden token prompt and retries once.
The token stays in this kernel’s memory; an empty response cancels the run.


In [ ]:
import os
from getpass import getpass

# Keep authentication interactive here, rather than in Gladius's batch runner.
# A failed archive request gets one chance to retry with a new token.
def run_with_mast_prompt(config):
    try:
        return run(config, progress=print)
    except PermissionError:
        if config.inputs.mode != "mast":
            raise

    print("MAST needs a valid token for an account with access to these images.")
    print("You can create one at https://auth.mast.stsci.edu/tokens")
    token = getpass("MAST token (hidden; Enter cancels): ").strip()
    if not token:
        print("Archive run cancelled. Local images do not need a MAST token.")
        return None

    # Store the token only in this kernel's environment, not in a notebook cell or file.
    previous_token = os.environ.get("MAST_API_TOKEN")
    os.environ["MAST_API_TOKEN"] = token
    del token

    try:
        return run(config, progress=print)
    except PermissionError:
        # Restore the previous value if the replacement did not authorize the request.
        if previous_token is None:
            os.environ.pop("MAST_API_TOKEN", None)
        else:
            os.environ["MAST_API_TOKEN"] = previous_token
        print("MAST still denied access. The token may be invalid or expired, or your account")
        print("may lack permission for these images. No further retry was made.")
        return None


# The returned path points to the run record. A cancelled or denied run returns None.
manifest_path = run_with_mast_prompt(config)

if manifest_path is not None:
    # Read the JSON record to inspect outputs and the work performed.
    manifest_text = manifest_path.read_text()
    manifest = json.loads(manifest_text)
    print("Manifest:", manifest_path)
    print("Work performed:", manifest["stage_counts"])
else:
    print("The review cells below need a successful run before they can show results.")


### Checking the results

The table below includes completed, skipped, and failed attempts. A completed
cutout with `accepted=False` failed the optional quality screen; Gladius still
saved its pixels. A `target_off_detector` entry means the target position was
outside that image. Processing errors are separate from quality flags.

The manifest records the run’s status, selected inputs, and errors. These records
help explain missing results; output files alone do not show that a run succeeded.


In [ ]:
from astropy.table import Table
from IPython.display import display
from gladius.inspection import audit_rows, contact_sheets, parent_preview

# A Python list of dictionaries: one row for every attempted input/target pair.
review = audit_rows(manifest_path)
audit = Table(rows=review)


# Show run outcomes and quality decisions separately from pixel-quality fractions.
display(audit[["target", "parent", "filter", "status", "accepted", "display_eligible", "axis_ratio", "reason"]])

# Show run outcomes and quality decisions separately from pixel-quality fractions.
display(audit[["target", "parent", "padding_fraction", "do_not_use_fraction", "saturated_fraction"]])

print("Run status:", manifest["status"])
print("Input images used:", len(manifest["sources"]))
print("Parents rejected by date/detector/filter:", manifest["selection_rejections"])
print("Recorded errors:", manifest["errors"])


### Comparing cutouts with the original image

The previews below draw each requested cutout boundary and target position on
the original image. A boundary can cross the detector edge when padding is on.
Purple pixels in a cutout preview mark unavailable values.

The crosses show the positions you supplied, rather than measured star centres.
The north and east arrows come from the image’s WCS. Comparing an image with its
own cutout checks placement within that image; it does not independently verify
its sky coordinates.

The next cell displays a chart file if you provide a path. Otherwise, it provides a link to
an interactive Aladin view centred on the target, starting with the 2MASS colour
survey. You can change the survey, field width, and target index in the cell.
The local notebook uses the ipyaladin widget to display the view inline. On Nexus,
`SHOW_ALADIN_INLINE = True` enables it; the default only generates a link.

Aladin loads survey images from IRSA around the selected target over
the network. For restricted target positions, a local chart avoids sending them
to that service. Neither option uploads your Roman image.


In [ ]:
# Several targets can share an original image. A set collects each image ID once.
completed_parent_ids = set()
for entry in manifest["entries"].values():
    if entry["status"] == "complete":
        completed_parent_ids.add(entry["parent_sha256"])

# Draw the completed cutout boundaries on each original image.
for parent_id in sorted(completed_parent_ids):
    preview = parent_preview(manifest_path, parent_id, width=800)
    display(preview)


In [ ]:
from urllib.parse import urlencode

from PIL import Image
from IPython.display import Markdown, display

# A file path displays your chart. None opens an Aladin sky view instead.
FINDING_CHART = None
# FINDING_CHART = Path("/path/to/finding-chart.png")

# Choose which configured target to show and the field width in degrees.
REFERENCE_TARGET_INDEX = 0
REFERENCE_FOV_DEG = 0.1
# IRSA hosts this mirror of the 2MASS colour survey.
REFERENCE_SURVEY = "https://irsa.ipac.caltech.edu/data/hips/CDS/2MASS/Color"
SHOW_ALADIN_INLINE = True  # True loads the sky view inside this notebook.

if FINDING_CHART is not None:
    # Resize only the displayed chart; leave the file and cutout pixels unchanged.
    with Image.open(FINDING_CHART) as chart:
        reference = chart.convert("RGB")
        reference.thumbnail((900, 900))
        display(reference)
else:
    target = config.targets[REFERENCE_TARGET_INDEX]

    if target.ra is None or target.dec is None:
        print("The Aladin view needs sky coordinates. This target uses image pixels.")
    else:
        # Aladin loads the selected survey around this target in your browser.
        # urlencode keeps coordinates and survey names safe in the URL.
        parameters = urlencode({
            "ra": target.ra,
            "dec": target.dec,
            "fov": REFERENCE_FOV_DEG,
            "hips": REFERENCE_SURVEY,
        })
        aladin_url = f"https://aladin.cds.unistra.fr/AladinLite/?{parameters}"

        display(Markdown(f"[Open this field in Aladin]({aladin_url})"))
        if SHOW_ALADIN_INLINE:
            from ipyaladin import Aladin

            # The widget opens at these coordinates without resolving a target name.
            aladin = Aladin(
                target=(target.ra, target.dec),
                fov=REFERENCE_FOV_DEG,
                survey=REFERENCE_SURVEY,
                height=600,
            )
            display(aladin)


### Reviewing every cutout

The contact sheets include all completed cutouts, including those the screen
flagged. Each cross marks a requested target position. The gold square shows
the region used by the quality screen, so defects outside it can go unnoticed.

The score measures elongation in the image’s autocorrelation: its similarity to
shifted copies of itself. Scores near one describe a rounder pattern; larger
scores describe a more elongated pattern. This is not a fitted stellar shape.
An insufficient-signal or insufficient-tile result means the screen could not
make a useful measurement, rather than that it detected trailing.

Images in the same target, filter, and unit group share a brightness stretch.
Different groups have separate stretches, so their displayed brightnesses do
not measure relative source fluxes.


In [ ]:
# Twelve images per PNG page; iterate every page to review all complete cutouts.
# Keep review images beside a folder identifying this run and revision.
review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
review_directory.mkdir(parents=True, exist_ok=True)

# Save and display every page, including cutouts flagged by the quality screen.
for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
    path = review_directory / f"all-cutouts-{page_number:03d}.png"
    sheet.save(path)
    display(sheet)
print("Review PNGs:", review_directory)


### Changing the quality screen

The next cell changes the screen settings and creates a revision of the run.
With cached cutouts available, Gladius can reassess their pixels without
extracting them again. The new manifest records the current decisions; the
original image files keep their creation records.

A score at or above the threshold gets flagged. A larger threshold tolerates
more elongation; `None` disables the screen. The tile size and maximum shift
also affect the measurement. These settings need checking against representative
images rather than treating the example values as a scientific calibration.

Flagged cutouts remain saved. `exclude_from_stacks` controls whether they
contribute to display products; it does not remove their flags. Gladius currently
has no manual keep-or-exclude control for individual exposures.


In [ ]:
# Edit these values after inspecting the first audit and contact sheets.
NEW_ACF_LIMIT = 2.0  # Demonstration only; choose based on representative images.
NEW_TILE_SIDE = config.quality.tile_size
NEW_LAG_RADIUS = config.quality.radius

# Copy the current settings into an editable dictionary.
settings = config.model_dump(mode="json")
settings["quality"].update(autocorrelation_limit=NEW_ACF_LIMIT,
                           tile_size=NEW_TILE_SIDE, radius=NEW_LAG_RADIUS)

# Validate the edited settings, then rerun using cached pixels where available.
review_config = RunConfig.model_validate(settings)
old_manifest_path = manifest_path
manifest_path = run(review_config, progress=print)
manifest = json.loads(manifest_path.read_text())

print("Previous revision:", old_manifest_path.parent)
print("Current revision:", manifest_path.parent)
print("Work performed:", manifest["stage_counts"])

# With the cache intact, 'extracted' is 0. Original revision files remain unchanged.
# Rebuild the review table and thumbnails from the new revision.
audit = Table(rows=audit_rows(manifest_path))
display(audit[["target", "parent", "filter", "accepted", "display_eligible", "axis_ratio", "reason"]])
review_directory = WORKSPACE / "review" / config.run.name / manifest_path.parent.name
review_directory.mkdir(parents=True, exist_ok=True)
for page_number, sheet in enumerate(contact_sheets(manifest_path), start=1):
    sheet.save(review_directory / f"all-cutouts-{page_number:03d}.png")
    display(sheet)

# Use the revised configuration as the starting point for the next section.
config = review_config


### Adding display images

A median stack combines aligned exposures. An animation shows the exposures
one at a time. A colour composite assigns three filter images to red, green,
and blue. These outputs are independent choices.

The next cells request all three kinds of display. A single-image example cannot
show noise reduction across exposures. A colour composite needs eligible images
in all three selected filters; the manifest explains any missing-filter skips.

These products are for inspection. Alignment interpolates pixels, and Gladius
does not match stellar image shapes, match photometric calibration, or propagate
uncertainties through stacking. Individual cutouts retain the original pixels
for measurements.


In [ ]:
# Copy the reviewed settings and enable the three independent display products.
settings = config.model_dump(mode="json")
settings["stacks"].update(enabled=True, gif_enabled=True, rgb_enabled=True,
                          blue_filter="F062", green_filter="F184", red_filter="F213")

# Check the new settings, then build displays using saved cutouts where possible.
config = RunConfig.model_validate(settings)
manifest_path = run(config, progress=print)
manifest = json.loads(manifest_path.read_text())

# The display manifest lists products, contributing images, and reasons for exclusions.
display_manifest_path = manifest_path.parent / "stack-manifest.json"
display_manifest = json.loads(display_manifest_path.read_text())
print("Native work:", manifest["stage_counts"])
for category in ("products", "animations", "composites", "composite_skips", "excluded"):
    print(category, display_manifest[category])


In [ ]:
from astropy.io import fits
from IPython.display import Image as DisplayImage
from gladius.inspection import image_preview

# Median images: display stretch only; the FITS science values stay unchanged.
for product in display_manifest["products"]:
    path = manifest_path.parent / product["path"]
    with fits.open(path) as hdul:
        print("Median:", product["target"], product["filter"], "inputs:", len(product["inputs"]))
        display(image_preview(hdul[0].data))

        # COVERAGE counts how many exposures contributed to each output pixel.
        coverage = hdul["COVERAGE"].data
        print("Contributing exposures per output pixel:", int(coverage.min()), "to", int(coverage.max()))

# The GIF is individual aligned exposures, NOT the median shown above.
for animation in display_manifest["animations"]:
    print("GIF:", animation["target"], animation["filter"], "frames:", len(animation["frames"]))
    display(DisplayImage(filename=str(manifest_path.parent / animation["path"])))

# RGB has independently stretched filter channels; it is not physical colour.
for composite in display_manifest["composites"]:
    print("RGB:", composite["target"], composite["channel_filters"])
    with Image.open(manifest_path.parent / composite["path"]) as image:
        display(image.resize((300, 300), Image.Resampling.NEAREST))


## Continuing with your own sample

The interface and Python functions use the same configuration model. A YAML file
saved from the interface can be read in Python with `load_config(path)`. Changes
in the interface do not change this notebook’s `config` variable automatically.

For archive work, `run(config, refresh_inputs=True)` refreshes the input list.
`run(config, fresh=True)` bypasses reuse and repeats the processing. Both can
require more archive access. Existing revisions keep their files.

Further details are in the [configuration guide](https://roman-gladius.readthedocs.io/en/latest/configuration.html),
[output guide](https://roman-gladius.readthedocs.io/en/latest/products.html), and
[scientific conventions](https://roman-gladius.readthedocs.io/en/latest/science.html).

This is an independent community tutorial by the Gladius contributors. The
repository’s `CITATION.cff` provides citation information. Notebook outputs can
contain restricted images and pointing information; a shared copy needs those
outputs cleared.
